# Quantum Simulator — Simulator-Generated ML Evaluation

This notebook runs the project simulator to create measurement observations from normal circuits, controlled program-gate mutations, and simulated readout noise. It trains and evaluates ML models with a **circuit-family-separated holdout split**.

**Interpretation:** this is a controlled simulator experiment with known intervention labels. It is not official Bugs4Q/QMutBench accuracy and is not a physical IBM hardware result. Official benchmark accuracy remains unavailable until independently verified diagnosis labels exist.


In [ ]:
# Fresh isolated Colab checkout. Existing project folders are not overwritten.
import subprocess, sys
REPO = 'https://github.com/ruckscode/Quantum_Simulator.git'
BRANCH = 'fix/statistics-module-collision'
from pathlib import Path
root = Path('/content/Quantum_Simulator_final_research')
if not root.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO,str(root)], check=True)
else:
    subprocess.run(['git','-C',str(root),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(root),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(root),'pull','--ff-only','origin',BRANCH], check=True)
%cd /content/Quantum_Simulator_final_research
%pip install -e . pandas matplotlib scikit-learn


## Run the simulator-generated dataset and ML evaluation

The script uses actual execution outputs from this repository's local statevector simulator. The injected scenario label is known by construction; do not describe this as independently verified benchmark ground truth.


In [ ]:
import subprocess, sys
run = subprocess.run([sys.executable, 'experiments/simulator_generated_ml_baseline.py', '--repetitions', '12', '--shots', '128', '--output-dir', 'reports/simulator_generated_ml'], text=True, capture_output=True)
print(run.stdout)
if run.stderr:
    print(run.stderr[-6000:])
if run.returncode != 0:
    raise RuntimeError(f'ML experiment failed with exit code {run.returncode}')


In [ ]:
import json, pandas as pd
from IPython.display import display, Image, Markdown
out = Path('reports/simulator_generated_ml')
metrics = pd.read_csv(out/'simulator_generated_ml_metrics.csv')
data = pd.read_csv(out/'simulator_generated_ml_dataset.csv')
summary = json.loads((out/'simulator_generated_ml_summary.json').read_text())
display(Markdown('## Model metrics'))
display(metrics.style.format({c:'{:.3f}' for c in ['accuracy','macro_precision','macro_recall','macro_f1']}))
display(Markdown('## Generated observations (preview)'))
display(data.head(10))
display(Markdown('## Accuracy and macro-F1'))
display(Image(filename=str(out/'simulator_generated_ml_metrics.png')))
display(Markdown('## Random Forest confusion matrix'))
display(Image(filename=str(out/'simulator_generated_ml_confusion_matrix.png')))
print('Summary:', json.dumps(summary, indent=2))


## Robust ML evaluation: family-held-out cross-validation\n\nThis second evaluation uses leave-one-circuit-family-out predictions and compares feature sets with and without TVD. It is a stress test for stability and shortcut reliance, still on controlled simulator-generated labels—not benchmark or IBM accuracy.\n

In [ ]:
import subprocess, sys, pandas as pd\nrobust = subprocess.run([sys.executable, 'experiments/robust_ml_evaluation.py', '--repetitions', '12', '--shots', '128', '--output-dir', 'reports/robust_ml_evaluation'], text=True, capture_output=True)\nprint(robust.stdout)\nif robust.stderr: print(robust.stderr[-6000:])\nif robust.returncode != 0: raise RuntimeError(f'Robust ML evaluation failed: {robust.returncode}')\nrobust_metrics = pd.read_csv('reports/robust_ml_evaluation/group_cv_metrics.csv')\ndisplay(robust_metrics.style.format({c:'{:.3f}' for c in ['accuracy','macro_precision','macro_recall','macro_f1']}))\ndisplay(Image(filename='reports/robust_ml_evaluation/feature_ablation_macro_f1.png'))\n

## Audit official benchmark labels (optional; no diagnosis labels are invented)\n\nDownload the official QMutBench CSV export from [the QMutBench repository](https://github.com/EnautMendi/QMutBench) and/or extract the official Bugs4Q archive from [Zenodo](https://zenodo.org/records/8148982). Then run the audit with the actual file paths. For example:\n\n`python experiments/benchmark_label_audit.py --qmutbench-csv /content/all_data.csv --bugs4q-root /content/Bugs4Q-Framework --output reports/benchmark_label_audit.json`\n\nThe audit streams large CSVs and reports schema, missingness, candidate metadata labels, and dataset file counts. QMutBench mutation/survival outcomes are not automatically hardware-versus-program diagnosis labels.\n

## How to interpret the output

- **Accuracy / macro-F1** describe classification of these controlled simulator-generated conditions only.
- The split holds out complete circuit families to reduce near-duplicate leakage.
- `SIMULATED_HARDWARE_NOISE` is injected through the local `HardwareModel`, not collected from IBM hardware.
- `PROGRAM_FAULT` is a known gate mutation created by the experiment.
- These metrics do not resolve the official benchmark-label problem. QMutBench mutation metadata and Bugs4Q bug-type metadata are not interchangeable with diagnosis-category ground truth.
- IBM access is not attempted and no quantum job is submitted by this notebook.


## IBM Quantum backend access check (no job submission)\n\nRun this cell only when you are ready to verify the account/instance. It loads the token from Colab Secrets, lists accessible backends, and does **not** submit a circuit. It never prints the token.\n

In [ ]:
# Backend access check only: no quantum job is submitted.\nimport os, json, subprocess, sys\nfrom pathlib import Path\ntry:\n    from google.colab import userdata\n    token = userdata.get('QISKIT_IBM_TOKEN')\nexcept Exception as exc:\n    raise RuntimeError('Could not read the QISKIT_IBM_TOKEN Colab Secret. Add/share the secret in Colab; do not paste it into notebook text.') from exc\nif not token:\n    raise RuntimeError('QISKIT_IBM_TOKEN secret is empty or unavailable. No token was printed.')\nos.environ['QISKIT_IBM_TOKEN'] = token\ntry:\n    instance = userdata.get('QISKIT_IBM_INSTANCE')\n    if instance: os.environ['QISKIT_IBM_INSTANCE'] = instance\nexcept Exception:\n    pass\ndel token\n%pip -q install qiskit qiskit-ibm-runtime\ncheck = subprocess.run([sys.executable, 'ibm_validation.py', '--list-backends', '--output', 'reports/ibm_access_check.json'], text=True, capture_output=True)\nif check.returncode != 0:\n    print(check.stderr[-4000:])\n    raise RuntimeError(f'Backend discovery command exited with {check.returncode}; no job was submitted.')\nreport = json.loads(Path('reports/ibm_access_check.json').read_text())\ndiscovery = report.get('backend_discovery', {})\nprint('Access check status:', discovery.get('status'))\nprint('Backend count:', len(discovery.get('backends', [])))\nprint('Backend names:', [b.get('name') for b in discovery.get('backends', [])])\nprint('Access details:', json.dumps(discovery.get('access', {}), indent=2))\nif discovery.get('error'): print('Discovery error:', discovery['error'])\nprint('No IBM circuit job was submitted.')\n